In [ ]:
#!/usr/bin/env python3
# hyper_mmca_bifurcation.py
"""
Bifurcation / hysteresis experiment for 3-uniform hypergraph + MMCA+Theta.

Usage:
    python hyper_mmca_bifurcation.py

What it does:
- Generate one random 3-uniform hypergraph per run (or reuse one).
- For a grid of beta values, run MMCA until convergence from two different initializations:
    * "low" init (rho0_small) -> forward branch
    * "high" init (rho0_large) -> backward branch (for hysteresis)
- Plot steady-state rho vs beta (forward/backward). Also plot example time series.
- Simple oscillation check via std of last window.

Adjust parameters in the PARAMETERS section.
"""

import numpy as np
import random
import math
import matplotlib.pyplot as plt
import pandas as pd
from time import time
from copy import deepcopy

# --------------------------- Hypergraph generator (unchanged) ---------------------------
def generate_hypergraph_H_n_m_3(n, kappa, seed=None, batch_size=3000):
    if seed is not None:
        np.random.seed(seed)
        random.seed(seed)
    d = 3
    m = int(round(n * kappa / d))
    max_possible = math.comb(n, d)
    if m > max_possible:
        raise ValueError("m > C(n,3)")
    nodes = np.arange(n)
    edges_set = set()
    while len(edges_set) < m:
        tris = np.random.choice(nodes, size=(batch_size, d), replace=True)
        for i in range(tris.shape[0]):
            row = tris[i]
            if len(np.unique(row)) < d:
                row = np.random.choice(nodes, size=(d,), replace=False)
            tris[i] = np.sort(row)
        for row in tris:
            if len(edges_set) >= m:
                break
            edges_set.add(tuple(row.tolist()))
    hyperedges = np.array(list(edges_set), dtype=int)
    hyperedges.sort(axis=1)
    return hyperedges

# --------------------------- MMCA model class (slightly adapted) ---------------------------
class HyperSISModel:
    def __init__(self, hyperedges, n):
        self.hyperedges = np.asarray(hyperedges, dtype=int)
        self.n = n
        self.m = self.hyperedges.shape[0]
        self.edge_i = self.hyperedges[:,0]
        self.edge_j = self.hyperedges[:,1]
        self.edge_k = self.hyperedges[:,2]

        self.node_edge_pos = [[] for _ in range(self.n)]
        for ei in range(self.m):
            a,b,c = self.hyperedges[ei]
            self.node_edge_pos[a].append((ei,0))
            self.node_edge_pos[b].append((ei,1))
            self.node_edge_pos[c].append((ei,2))

    def _compute_edge_products(self, p):
        p_i = p[self.edge_i]; p_j = p[self.edge_j]; p_k = p[self.edge_k]
        return (p_j * p_k, p_i * p_k, p_i * p_j)

    def _update_theta_from_p(self, p, Theta, eta, gamma):
        p_i = p[self.edge_i]; p_j = p[self.edge_j]; p_k = p[self.edge_k]
        Phi = p_i * p_j * p_k
        Theta_new = Theta * (1.0 - eta * Phi) + gamma * (1.0 - Theta)
        return np.clip(Theta_new, 0.0, 1.0), Phi

    def mmca_step(self, p, Theta, beta_d, mu):
        """One MMCA step: compute Q, then p_next (vectorized-ish), return p_next, Theta_next, Phi."""
        prod_pos0, prod_pos1, prod_pos2 = self._compute_edge_products(p)
        Q = np.ones(self.n, dtype=float)
        # per-node accumulation
        for node in range(self.n):
            prod_val = 1.0
            for (ei,pos) in self.node_edge_pos[node]:
                if pos == 0:
                    prod_oth = prod_pos0[ei]
                elif pos == 1:
                    prod_oth = prod_pos1[ei]
                else:
                    prod_oth = prod_pos2[ei]
                prod_val *= (1.0 - Theta[ei] * beta_d * prod_oth)
            Q[node] = prod_val
        p_next = (1.0 - p) * (1.0 - Q) + (1.0 - mu) * p
        Theta_next, Phi = self._update_theta_from_p(p, Theta, eta=global_params['eta'], gamma=global_params['gamma'])
        return p_next, Theta_next, Q, Phi

    def run_mmca_until_converged(self, beta_d, mu, eta, gamma,
                                 rho0=0.01, Theta0=1.0,
                                 tol=1e-6, max_iter=2000, record_traj=False):
        """
        Iterate MMCA starting from p_i(0)=rho0 and Theta_e(0)=Theta0 until convergence (inf-norm change < tol)
        Returns: steady_rho (scalar), steady_Theta_mean (scalar), optional full trajectories (if record_traj)
        """
        # init
        p = np.full(self.n, float(rho0), dtype=float)
        Theta = np.full(self.m, float(Theta0), dtype=float)
        if record_traj:
            rho_hist = [p.mean()]
            theta_hist = [Theta.mean()]

        for it in range(max_iter):
            p_next, Theta_next, Q, Phi = self.mmca_step(p, Theta, beta_d, mu)
            dp = np.max(np.abs(p_next - p))
            dTheta = np.max(np.abs(Theta_next - Theta))
            p, Theta = p_next, Theta_next
            if record_traj:
                rho_hist.append(p.mean()); theta_hist.append(Theta.mean())
            if dp < tol and dTheta < tol:
                # converged
                break
        # return steady statistics
        if record_traj:
            return p.mean(), Theta.mean(), np.array(rho_hist), np.array(theta_hist), it+1
        else:
            return p.mean(), Theta.mean(), it+1

# --------------------------- Experiment parameters ---------------------------
# You can tune these values
n = 1000
kappa = 6.0
mu = 0.2
T_example = 400   # length of example time series to show
seed_base = 12345

# sweep parameters
beta_min = 0.01
beta_max = 0.6
beta_steps = 41   # => step ~ (beta_max-beta_min)/(steps-1)
beta_list = np.linspace(beta_min, beta_max, beta_steps)

# initial conditions for forward/backward branches
rho0_small = 1e-4   # near zero seed for forward scan
rho0_large = 0.5    # large seed for backward scan

# MMCA iteration params
tol = 1e-8
max_iter = 2000

# global params for Theta update accessible to model.mmca_step
global_params = {'eta': 0.8, 'gamma': 0.02}

# how many different random graphs to sample (independent replicates)
R_graphs = 8  # moderate number for stability; increase if you have CPU
reuse_same_graph = False  # if True, use same graph for all betas (helps see bifurcation on same topology)

# --------------------------- experiment: forward/backward sweep ---------------------------
def sweep_bifurcation(n, kappa, beta_list, mu, eta, gamma,
                      rho0_small, rho0_large, tol, max_iter,
                      R_graphs=5, reuse_same_graph=False, seed_base=12345):
    """For each graph (R_graphs), produce forward and backward steady rho arrays over beta_list.
       Returns lists of arrays: forward_runs (R x len(beta_list)), backward_runs similarly.
    """
    forward_runs = []
    backward_runs = []
    times_taken = []

    for r in range(R_graphs):
        gen_seed = None if seed_base is None else (seed_base + r)
        if reuse_same_graph and r > 0:
            # to reuse, just reuse previous hyperedges
            pass
        hyperedges = generate_hypergraph_H_n_m_3(n=n, kappa=kappa, seed=gen_seed, batch_size=3000)
        model = HyperSISModel(hyperedges, n)

        forward = np.zeros(len(beta_list))
        backward = np.zeros(len(beta_list))

        # forward scan: increasing beta, start each run from small rho initial (we can also warm-start from previous)
        p_start = None
        Theta_start = None
        t0 = time()
        for i, beta in enumerate(beta_list):
            # run from small seed (or warm-start)
            if i == 0:
                rho0 = rho0_small
            else:
                # warm-start with previous converged p to speed up, but keep logic: this explores stable branch
                # set initial p to small seed again to follow low-branch; to reveal saddle-node you should both do warm-starts and resets.
                rho0 = rho0_small
            steady_rho, steady_Theta_mean, iters = model.run_mmca_until_converged(
                beta_d=beta, mu=mu, eta=eta, gamma=gamma,
                rho0=rho0, Theta0=1.0, tol=tol, max_iter=max_iter, record_traj=False
            )
            forward[i] = steady_rho

        # backward scan: decreasing beta, start from large seed to try to find high-branch
        for i_rev, beta in enumerate(beta_list[::-1]):
            if i_rev == 0:
                rho0 = rho0_large
            else:
                rho0 = rho0_large
            steady_rho, steady_Theta_mean, iters = model.run_mmca_until_converged(
                beta_d=beta, mu=mu, eta=eta, gamma=gamma,
                rho0=rho0, Theta0=1.0, tol=tol, max_iter=max_iter, record_traj=False
            )
            backward[len(beta_list)-1 - i_rev] = steady_rho

        forward_runs.append(forward)
        backward_runs.append(backward)
        times_taken.append(time()-t0)
        print(f"[graph {r+1}/{R_graphs}] done, time {times_taken[-1]:.1f}s")

    return np.array(forward_runs), np.array(backward_runs), times_taken

# --------------------------- run the sweep ---------------------------
print("Running bifurcation sweep (forward/backward) ...")
t0_all = time()
forward_runs, backward_runs, times = sweep_bifurcation(
    n=n, kappa=kappa, beta_list=beta_list, mu=mu,
    eta=global_params['eta'], gamma=global_params['gamma'],
    rho0_small=rho0_small, rho0_large=rho0_large,
    tol=tol, max_iter=max_iter, R_graphs=R_graphs, reuse_same_graph=reuse_same_graph,
    seed_base=seed_base
)
print("Sweep finished in {:.1f}s".format(time()-t0_all))

# --------------------------- Analysis & plotting ---------------------------
# average across graphs
mean_forward = forward_runs.mean(axis=0)
std_forward = forward_runs.std(axis=0)
mean_backward = backward_runs.mean(axis=0)
std_backward = backward_runs.std(axis=0)

# simple indicator of bistability: |forward - backward|
diff_branch = np.abs(mean_forward - mean_backward)

# plot bifurcation diagram (forward/backward)
plt.figure(figsize=(8,5))
plt.fill_between(beta_list, mean_forward-std_forward, mean_forward+std_forward, alpha=0.2, color='C0')
plt.plot(beta_list, mean_forward, '-o', color='C0', label='forward (low init)')
plt.fill_between(beta_list, mean_backward-std_backward, mean_backward+std_backward, alpha=0.2, color='C1')
plt.plot(beta_list, mean_backward, '-s', color='C1', label='backward (high init)')
plt.xlabel(r'$\beta$')
plt.ylabel(r'Steady-state $\rho$')
plt.title('Bifurcation diagram (MMCA on random 3-uniform hypergraphs)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# plot difference (hysteresis measure)
plt.figure(figsize=(6,3))
plt.plot(beta_list, diff_branch, '-k')
plt.xlabel(r'$\beta$')
plt.ylabel(r'$|\rho_{fw}-\rho_{bw}|$')
plt.title('Forward-backward difference (hysteresis measure)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()




In [ ]:
# --------------------------- plot example time series near suspected bistability ---------------------------
# choose representative beta values: pick where diff_branch is largest (candidate bistability region)
idx_peak = int(np.argmax(diff_branch))
beta_peak = beta_list[idx_peak]
beta_before = beta_list[max(0, idx_peak-3)]
beta_after = beta_list[min(len(beta_list)-1, idx_peak+3)]
example_betas = [beta_before, beta_peak, beta_after]

print("Example betas chosen for time series:", example_betas)

# For each example beta, simulate two trajectories from low and high initial conditions and plot
fig, axes = plt.subplots(len(example_betas), 2, figsize=(10, 3*len(example_betas)), sharex=True)
for i, bval in enumerate(example_betas):
    # single fresh graph for time-series clarity
    hyperedges = generate_hypergraph_H_n_m_3(n=n, kappa=kappa, seed=seed_base + 100 + i, batch_size=3000)
    model = HyperSISModel(hyperedges, n)
    # simulate from low initial
    rho_low, theta_low, rho_hist_low, theta_hist_low, iters_low = model.run_mmca_until_converged(
        beta_d=bval, mu=mu, eta=global_params['eta'], gamma=global_params['gamma'],
        rho0=rho0_small, Theta0=1.0, tol=1e-9, max_iter=T_example, record_traj=True
    )
    # simulate from high initial
    rho_high, theta_high, rho_hist_high, theta_hist_high, iters_high = model.run_mmca_until_converged(
        beta_d=bval, mu=mu, eta=global_params['eta'], gamma=global_params['gamma'],
        rho0=rho0_large, Theta0=1.0, tol=1e-9, max_iter=T_example, record_traj=True
    )

    ts_low = np.arange(len(rho_hist_low))
    ts_high = np.arange(len(rho_hist_high))

    ax1 = axes[i,0]
    ax2 = axes[i,1]
    ax1.plot(ts_low, rho_hist_low, '-o', markersize=3, label=f'low init -> rho*={rho_low:.3f}')
    ax1.plot(ts_high, rho_hist_high, '-s', markersize=3, label=f'high init -> rho*={rho_high:.3f}')
    ax1.set_ylabel(r'$\rho(t)$')
    ax1.set_title(f'Beta={bval:.3f}  time series')
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(ts_low, theta_hist_low, '-o', markersize=3, label='Theta (low init)')
    ax2.plot(ts_high, theta_hist_high, '-s', markersize=3, label='Theta (high init)')
    ax2.set_ylabel(r'$\langle\Theta\rangle(t)$')
    ax2.set_title(f'Beta={bval:.3f}  Theta')
    ax2.legend()
    ax2.grid(alpha=0.3)

plt.xlabel('time step')
plt.tight_layout()
plt.show()

# --------------------------- simple oscillation check ---------------------------
# For a chosen beta (e.g. beta_peak), check last window std to flag oscillation
print("Oscillation diagnostic (std of last window):")
for arr, lab in [(forward_runs, 'forward_runs'), (backward_runs, 'backward_runs')]:
    std_vals = []
    for r in range(arr.shape[0]):
        # use a fresh long trajectory for this beta to detect oscillation
        hyperedges = generate_hypergraph_H_n_m_3(n=n, kappa=kappa, seed=seed_base + 500 + r, batch_size=3000)
        model = HyperSISModel(hyperedges, n)
        rho_s, theta_s, rho_hist, theta_hist, iters = model.run_mmca_until_converged(
            beta_d=beta_peak, mu=mu, eta=global_params['eta'], gamma=global_params['gamma'],
            rho0=rho0_large, Theta0=1.0, tol=1e-9, max_iter=2000, record_traj=True
        )
        last_window = rho_hist[-200:] if rho_hist.size > 200 else rho_hist
        std_last = float(np.std(last_window))
        std_vals.append(std_last)
    std_vals = np.array(std_vals)
    print(f"  {lab}: mean std(last window) = {std_vals.mean():.6f}, max = {std_vals.max():.6f}")

# --------------------------- save results ---------------------------
pd.DataFrame(forward_runs).to_csv("./forward_runs_mmca.csv", index=False, header=False)
pd.DataFrame(backward_runs).to_csv("./backward_runs_mmca.csv", index=False, header=False)
pd.DataFrame({'beta': beta_list, 'mean_fw': mean_forward, 'mean_bw': mean_backward,
              'diff': diff_branch}).to_csv("./bifurcation_summary.csv", index=False)

print("Saved forward/backward runs and bifurcation_summary.csv")